# BDLegalRAG — Stage 5: Clean, Reproducible Retrieval Evaluation

This notebook is the cleaned Stage 5 pipeline based on the validated analysis.

### Experiments
1. Raw BM25
2. Context-enriched BM25
3. Raw BGE-M3 Dense
4. Context-enriched BGE-M3 Dense
5. Raw Dense + Raw BM25 RRF
6. Context-enriched Dense + Context-enriched BM25 RRF
7. RRF sanity check

### Fixed paths
**Kaggle input dataset:**
`/kaggle/input/datasets/tanmimalamnuhin/for-new-stage-5/`

Required files:
- `bdlegalrag_corpus_v1.json`
- `gold_qa_set_v1.csv`
- `embedding_benchmark_cache.pkl`

**Kaggle working output:**
`/kaggle/working/BDLegalRAG/stage5_output/`

The notebook does not tune a method to force a higher score. All reported methods use the same 180-question gold set and the same 1,575-section corpus.


In [16]:
# ============================================================
# CELL 1 — Fixed paths + installation
# ============================================================

from pathlib import Path
import sys
import subprocess

INPUT_DIR = Path("/kaggle/input/datasets/tanmimalamnuhin/for-new-stage-5")
OUT_DIR = Path("/kaggle/working/BDLegalRAG/stage5_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = INPUT_DIR / "bdlegalrag_corpus_v1.json"
GOLD_PATH = INPUT_DIR / "gold_qa_set_v1.csv"
EMBEDDING_CACHE_PATH = INPUT_DIR / "embedding_benchmark_cache.pkl"

required_files = [CORPUS_PATH, GOLD_PATH, EMBEDDING_CACHE_PATH]

print("INPUT_DIR :", INPUT_DIR)
print("OUT_DIR   :", OUT_DIR)

for p in required_files:
    print(f"{p.name}: {'FOUND' if p.exists() else 'MISSING'}")
    if not p.exists():
        raise FileNotFoundError(f"Required file not found: {p}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "rank_bm25"],
    check=True
)

print("\n✓ Paths validated and rank_bm25 installed.")


INPUT_DIR : /kaggle/input/datasets/tanmimalamnuhin/for-new-stage-5
OUT_DIR   : /kaggle/working/BDLegalRAG/stage5_output
bdlegalrag_corpus_v1.json: FOUND
gold_qa_set_v1.csv: FOUND
embedding_benchmark_cache.pkl: FOUND

✓ Paths validated and rank_bm25 installed.


In [17]:
# ============================================================
# CELL 2 — Imports + load corpus, Gold QA, embedding cache
# ============================================================

import gc
import hashlib
import json
import pickle
import re
import unicodedata

import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

try:
    import faiss
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"],
        check=True
    )
    import faiss

EMBED_MODEL_NAME = "BAAI/bge-m3"

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    corpus = json.load(f)

gold = pd.read_csv(GOLD_PATH)

with open(EMBEDDING_CACHE_PATH, "rb") as f:
    embedding_cache = pickle.load(f)

print("Corpus keys:", list(corpus.keys()))
print("Gold rows:", len(gold))
print("Embedding cache keys:", list(embedding_cache.keys()))

if len(gold) != 180:
    raise ValueError(f"Expected 180 Gold QA rows, found {len(gold)}")

if "bge_m3" not in embedding_cache:
    raise KeyError("bge_m3 not found in embedding_benchmark_cache.pkl")

raw_bge_embeddings = np.asarray(
    embedding_cache["bge_m3"],
    dtype=np.float32
)

print("Raw BGE-M3 cache shape:", raw_bge_embeddings.shape)


Corpus keys: ['meta', 'acts']
Gold rows: 180
Embedding cache keys: ['bge_m3', 'bangla_sbert', 'multilingual_e5_large', 'labse', 'mpnet_multilingual']
Raw BGE-M3 cache shape: (1575, 1024)


In [18]:
# ============================================================
# CELL 3 — Build the 1,575 legal-section chunks
# ============================================================

acts = corpus["acts"]
chunks = []

for act in acts:
    for section in act.get("sections", []):
        chunks.append({
            "act_id": act.get("act_id", ""),
            "act_title": act.get("act_title", ""),
            "act_no": act.get("act_no", ""),
            "act_year": act.get("act_year", ""),
            "source_url": act.get("source_url", ""),
            "section_number": section.get("section_number", ""),
            "section_title": section.get("section_title", ""),
            "section_text": section.get("section_text", ""),
        })

if len(acts) != 49:
    raise ValueError(f"Expected 49 acts, found {len(acts)}")

if len(chunks) != 1575:
    raise ValueError(f"Expected 1575 chunks, found {len(chunks)}")

if any(not str(c["section_text"]).strip() for c in chunks):
    raise ValueError("At least one chunk has empty section_text")

raw_texts = [c["section_text"] for c in chunks]

print("Acts:", len(acts))
print("Chunks:", len(chunks))
print("Empty section_text:", sum(not str(x).strip() for x in raw_texts))
print("\nFirst chunk:")
print(chunks[0])


Acts: 49
Chunks: 1575
Empty section_text: 0

First chunk:
{'act_id': '679', 'act_title': 'ভূমি-খতিয়ান (পার্বত্য চট্টগ্রাম) অধ্যাদেশ, ১৯৮৪', 'act_no': '2', 'act_year': '1985', 'source_url': 'http://bdlaws.minlaw.gov.bd/act-print-679.html', 'section_number': '1', 'section_title': '', 'section_text': '১৷ (১) এই অধ্যাদেশ ১৯৮৪ সনের ভূমি-খতিয়ান (পার্বত্য চট্টগ্রাম) অধ্যাদেশ নামে অভিহিত হইবে৷(২) ইহা সমগ্র পাবর্ত্য চট্টগ্রাম এলাকায় প্রযোজ্য হইবে৷'}


In [19]:
# ============================================================
# CELL 4 — Gold → corpus mapping
# ============================================================

def norm_text(x):
    if pd.isna(x):
        return ""
    x = unicodedata.normalize("NFKC", str(x).strip())
    x = re.sub(r"\s+", " ", x)
    return x.casefold()

def norm_section(x):
    if pd.isna(x):
        return ""

    x = unicodedata.normalize("NFKC", str(x).strip())
    x = x.replace("§", "").strip()

    arabic_to_bangla = str.maketrans(
        "0123456789",
        "০১২৩৪৫৬৭৮৯"
    )
    x = x.translate(arabic_to_bangla)
    x = re.sub(r"\s+", "", x)

    return x.casefold()

gold["__act_norm"] = gold["act_title"].map(norm_text)
gold["__sec_norm"] = gold["section_number"].map(norm_section)

gold_to_idx = {}

for i, c in enumerate(chunks):
    key = (
        norm_text(c["act_title"]),
        norm_section(c["section_number"])
    )
    if key[0] and key[1]:
        gold_to_idx.setdefault(key, []).append(i)

gold_indices = []

for _, row in gold.iterrows():
    key = (row["__act_norm"], row["__sec_norm"])
    matches = gold_to_idx.get(key, [])
    gold_indices.append(matches[0] if matches else None)

gold["gold_corpus_idx"] = gold_indices

matched = gold["gold_corpus_idx"].notna().sum()
print("Gold rows:", len(gold))
print("Matched:", matched)
print("Unmatched:", len(gold) - matched)

if matched != 180:
    missing = gold.loc[gold["gold_corpus_idx"].isna(),
                       ["qa_id", "act_title", "section_number"]]
    print(missing.head(20).to_string(index=False))
    raise ValueError("Gold mapping is not 180/180.")

gold_idx = gold["gold_corpus_idx"].astype(int).to_numpy()
queries = gold["question"].fillna("").astype(str).tolist()

print("✓ Gold mapping successful: 180/180")


Gold rows: 180
Matched: 180
Unmatched: 0
✓ Gold mapping successful: 180/180


In [20]:
# ============================================================
# CELL 5 — Build the exact context-enriched representation
# ============================================================

context_enriched_texts = []

for c in chunks:
    context_enriched_texts.append(
        f"Act Title: {c['act_title']}\n"
        f"Act No: {c['act_no']}\n"
        f"Act Year: {c['act_year']}\n"
        f"Section: {c['section_number']}\n"
        f"Section Title: {c['section_title']}\n"
        f"Text: {c['section_text']}"
    )

if len(context_enriched_texts) != 1575:
    raise ValueError("Context-enriched text count is not 1575.")

print("Context-enriched texts:", len(context_enriched_texts))
print("\nExample:")
print(context_enriched_texts[0][:1000])


Context-enriched texts: 1575

Example:
Act Title: ভূমি-খতিয়ান (পার্বত্য চট্টগ্রাম) অধ্যাদেশ, ১৯৮৪
Act No: 2
Act Year: 1985
Section: 1
Section Title: 
Text: ১৷ (১) এই অধ্যাদেশ ১৯৮৪ সনের ভূমি-খতিয়ান (পার্বত্য চট্টগ্রাম) অধ্যাদেশ নামে অভিহিত হইবে৷(২) ইহা সমগ্র পাবর্ত্য চট্টগ্রাম এলাকায় প্রযোজ্য হইবে৷


In [21]:
# ============================================================
# CELL 6 — Generate fresh context-enriched BGE-M3 embeddings
# ============================================================

import torch

CONTEXT_EMBED_PATH = OUT_DIR / "context_enriched_embeddings_cache.pkl"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Embedding model:", EMBED_MODEL_NAME)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free_mem, total_mem = torch.cuda.mem_get_info()
    print(f"Free GPU memory: {free_mem / 1024**3:.2f} GB")
    print(f"Total GPU memory: {total_mem / 1024**3:.2f} GB")

context_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

context_embeddings = context_model.encode(
    context_enriched_texts,
    batch_size=4,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

if context_embeddings.shape != (1575, 1024):
    raise ValueError(
        f"Unexpected context embedding shape: {context_embeddings.shape}"
    )

with open(CONTEXT_EMBED_PATH, "wb") as f:
    pickle.dump(context_embeddings, f)

print("\n✓ Context-enriched embeddings generated.")
print("Shape:", context_embeddings.shape)
print("Saved:", CONTEXT_EMBED_PATH)

del context_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Embedding model: BAAI/bge-m3
GPU: Tesla T4
Free GPU memory: 14.41 GB
Total GPU memory: 14.56 GB


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/394 [00:00<?, ?it/s]


✓ Context-enriched embeddings generated.
Shape: (1575, 1024)
Saved: /kaggle/working/BDLegalRAG/stage5_output/context_enriched_embeddings_cache.pkl


In [22]:
# ============================================================
# CELL 7 — Build raw and context-enriched dense indices
# ============================================================

def normalized_matrix(x):
    x = np.ascontiguousarray(np.asarray(x, dtype=np.float32))
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norms, 1e-12)

raw_dense_matrix = normalized_matrix(raw_bge_embeddings)
context_dense_matrix = normalized_matrix(context_embeddings)

if raw_dense_matrix.shape != (1575, 1024):
    raise ValueError(f"Unexpected raw BGE-M3 shape: {raw_dense_matrix.shape}")

if context_dense_matrix.shape != (1575, 1024):
    raise ValueError(
        f"Unexpected context BGE-M3 shape: {context_dense_matrix.shape}"
    )

raw_dense_index = faiss.IndexFlatIP(1024)
raw_dense_index.add(raw_dense_matrix)

context_dense_index = faiss.IndexFlatIP(1024)
context_dense_index.add(context_dense_matrix)

print("Raw dense index:", raw_dense_index.ntotal)
print("Context dense index:", context_dense_index.ntotal)


Raw dense index: 1575
Context dense index: 1575


In [23]:
# ============================================================
# CELL 8 — Query embeddings
# ============================================================

query_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

query_embeddings = query_model.encode(
    queries,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

if query_embeddings.shape != (180, 1024):
    raise ValueError(
        f"Unexpected query embedding shape: {query_embeddings.shape}"
    )

print("Query embedding shape:", query_embeddings.shape)

del query_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Query embedding shape: (180, 1024)


In [24]:
# ============================================================
# CELL 9 — Raw BM25 + Context-enriched BM25
# ============================================================

def tokenize(text):
    return re.findall(
        r"\w+",
        str(text).lower(),
        flags=re.UNICODE
    )

raw_tokenized_corpus = [tokenize(x) for x in raw_texts]
context_tokenized_corpus = [tokenize(x) for x in context_enriched_texts]
tokenized_queries = [tokenize(x) for x in queries]

raw_bm25 = BM25Okapi(raw_tokenized_corpus)
context_bm25 = BM25Okapi(context_tokenized_corpus)

print("Raw BM25 ready.")
print("Context-enriched BM25 ready.")


Raw BM25 ready.
Context-enriched BM25 ready.


In [25]:
# ============================================================
# CELL 10 — Retrieval functions
# ============================================================

def dense_rankings(index, q_embeddings, k=20):
    scores, idx = index.search(q_embeddings, k)
    return idx.astype(int), scores

def bm25_rankings(model, tokenized_q, k=20):
    all_idx = []
    all_scores = []

    for toks in tokenized_q:
        scores = model.get_scores(toks)
        top = np.argsort(scores)[::-1][:k]
        all_idx.append(top)
        all_scores.append(scores[top])

    return np.asarray(all_idx, dtype=int), np.asarray(all_scores)

def rrf_fuse(rank_a, rank_b, k=60, topk=20,
             weight_a=1.0, weight_b=1.0):
    fused_all = []

    for a, b in zip(rank_a, rank_b):
        scores = {}

        for r, idx in enumerate(a):
            idx = int(idx)
            scores[idx] = scores.get(idx, 0.0) + (
                weight_a / (k + r + 1)
            )

        for r, idx in enumerate(b):
            idx = int(idx)
            scores[idx] = scores.get(idx, 0.0) + (
                weight_b / (k + r + 1)
            )

        ranked = sorted(
            scores.items(),
            key=lambda z: z[1],
            reverse=True
        )

        fused_all.append([idx for idx, _ in ranked[:topk]])

    return np.asarray(fused_all, dtype=int)

print("✓ Retrieval functions ready.")


✓ Retrieval functions ready.


In [26]:
# ============================================================
# CELL 11 — Evaluation functions
# ============================================================

def evaluate_rankings(rankings, gold_indices=gold_idx):
    reciprocal_ranks = []
    recall5, recall10, recall20 = [], [], []
    precision5, precision10, precision20 = [], [], []

    for ranks, target in zip(rankings, gold_indices):
        ranks = list(map(int, ranks))
        target = int(target)

        if target in ranks:
            rank = ranks.index(target) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

        for k, rec_list, prec_list in [
            (5, recall5, precision5),
            (10, recall10, precision10),
            (20, recall20, precision20)
        ]:
            hit = target in ranks[:k]
            rec_list.append(float(hit))
            prec_list.append(float(hit) / k)

    return {
        "MRR": np.mean(reciprocal_ranks),
        "Recall@5": np.mean(recall5),
        "Precision@5": np.mean(precision5),
        "Recall@10": np.mean(recall10),
        "Precision@10": np.mean(precision10),
        "Recall@20": np.mean(recall20),
        "Precision@20": np.mean(precision20),
    }

print("✓ Evaluation function ready.")


✓ Evaluation function ready.


In [27]:
# ============================================================
# CELL 12 — Run the six core retrieval experiments
# ============================================================

raw_dense_idx, raw_dense_scores = dense_rankings(
    raw_dense_index, query_embeddings, k=20
)

context_dense_idx, context_dense_scores = dense_rankings(
    context_dense_index, query_embeddings, k=20
)

raw_bm25_idx, raw_bm25_scores = bm25_rankings(
    raw_bm25, tokenized_queries, k=20
)

context_bm25_idx, context_bm25_scores = bm25_rankings(
    context_bm25, tokenized_queries, k=20
)

raw_hybrid_idx = rrf_fuse(
    raw_bm25_idx,
    raw_dense_idx,
    k=60,
    topk=20
)

context_hybrid_idx = rrf_fuse(
    context_bm25_idx,
    context_dense_idx,
    k=60,
    topk=20
)

results = {
    "Raw Dense BGE-M3": evaluate_rankings(raw_dense_idx),
    "Raw BM25": evaluate_rankings(raw_bm25_idx),
    "Raw Hybrid RRF 1:1": evaluate_rankings(raw_hybrid_idx),
    "Context-enriched Dense BGE-M3": evaluate_rankings(context_dense_idx),
    "Context-enriched BM25": evaluate_rankings(context_bm25_idx),
    "Context-enriched Hybrid RRF 1:1": evaluate_rankings(
        context_hybrid_idx
    ),
}

results_df = pd.DataFrame(results).T

print(
    results_df[
        ["MRR", "Recall@5", "Recall@10", "Recall@20"]
    ].round(4).to_string()
)


                                    MRR  Recall@5  Recall@10  Recall@20
Raw Dense BGE-M3                 0.6906    0.8111     0.8722     0.9278
Raw BM25                         0.1705    0.2444     0.2889     0.3556
Raw Hybrid RRF 1:1               0.3900    0.6111     0.7722     0.8722
Context-enriched Dense BGE-M3    0.8087    0.9389     0.9667     0.9833
Context-enriched BM25            0.2096    0.2889     0.3889     0.4333
Context-enriched Hybrid RRF 1:1  0.4585    0.7056     0.9056     0.9722


In [28]:
# ============================================================
# CELL 13 — RRF sanity check
# ============================================================

dense_plus_dense = rrf_fuse(
    context_dense_idx,
    context_dense_idx,
    k=60,
    topk=20
)

direct = evaluate_rankings(context_dense_idx)
fused = evaluate_rankings(dense_plus_dense)

print("Context Dense:")
print(pd.Series(direct).round(6))

print("\nContext Dense + Context Dense RRF:")
print(pd.Series(fused).round(6))

if not np.array_equal(context_dense_idx, dense_plus_dense):
    raise AssertionError(
        "RRF sanity check failed: Dense + Dense rankings differ."
    )

for key in direct:
    if not np.isclose(direct[key], fused[key]):
        raise AssertionError(
            f"RRF sanity check failed for {key}."
        )

print("\n✓ RRF implementation passed the sanity check.")


Context Dense:
MRR             0.808686
Recall@5        0.938889
Precision@5     0.187778
Recall@10       0.966667
Precision@10    0.096667
Recall@20       0.983333
Precision@20    0.049167
dtype: float64

Context Dense + Context Dense RRF:
MRR             0.808686
Recall@5        0.938889
Precision@5     0.187778
Recall@10       0.966667
Precision@10    0.096667
Recall@20       0.983333
Precision@20    0.049167
dtype: float64

✓ RRF implementation passed the sanity check.


In [29]:
# ============================================================
# CELL 14 — Save final Stage 5 artifacts
# ============================================================

final_results = results_df.reset_index().rename(
    columns={"index": "Method"}
)

final_results.to_csv(
    OUT_DIR / "stage5_final_comparison.csv",
    index=False
)

def save_rankings(path, rankings, method_name):
    records = []

    for pos, row in gold.reset_index(drop=True).iterrows():
        for rank, idx in enumerate(rankings[pos], start=1):
            records.append({
                "qa_id": row["qa_id"],
                "question": row["question"],
                "method": method_name,
                "rank": rank,
                "corpus_idx": int(idx),
            })

    pd.DataFrame(records).to_csv(path, index=False)

ranking_outputs = {
    "stage5_raw_dense_top20.csv":
        (raw_dense_idx, "Raw Dense BGE-M3"),

    "stage5_raw_bm25_top20.csv":
        (raw_bm25_idx, "Raw BM25"),

    "stage5_raw_hybrid_rrf_top20.csv":
        (raw_hybrid_idx, "Raw Hybrid RRF 1:1"),

    "stage5_context_dense_top20.csv":
        (context_dense_idx, "Context-enriched Dense BGE-M3"),

    "stage5_context_bm25_top20.csv":
        (context_bm25_idx, "Context-enriched BM25"),

    "stage5_context_hybrid_rrf_top20.csv":
        (context_hybrid_idx, "Context-enriched Hybrid RRF 1:1"),
}

for filename, (ranking, method_name) in ranking_outputs.items():
    save_rankings(
        OUT_DIR / filename,
        ranking,
        method_name
    )

with open(OUT_DIR / "stage5_context_enriched_embeddings_cache.pkl", "wb") as f:
    pickle.dump(context_embeddings, f)

# Input hashes for reproducibility
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

hash_rows = []
for name, path in {
    "corpus": CORPUS_PATH,
    "gold_qa": GOLD_PATH,
    "embedding_cache": EMBEDDING_CACHE_PATH,
}.items():
    hash_rows.append({
        "file": name,
        "path": str(path),
        "sha256": sha256_file(path)
    })

pd.DataFrame(hash_rows).to_csv(
    OUT_DIR / "stage5_input_file_hashes.csv",
    index=False
)

print("=== FINAL STAGE 5 RESULTS ===")
print(final_results.round(4).to_string(index=False))

print("\nOutput directory:")
print(OUT_DIR)

print("\n✓ All Stage 5 artifacts saved.")


=== FINAL STAGE 5 RESULTS ===
                         Method    MRR  Recall@5  Precision@5  Recall@10  Precision@10  Recall@20  Precision@20
               Raw Dense BGE-M3 0.6906    0.8111       0.1622     0.8722        0.0872     0.9278        0.0464
                       Raw BM25 0.1705    0.2444       0.0489     0.2889        0.0289     0.3556        0.0178
             Raw Hybrid RRF 1:1 0.3900    0.6111       0.1222     0.7722        0.0772     0.8722        0.0436
  Context-enriched Dense BGE-M3 0.8087    0.9389       0.1878     0.9667        0.0967     0.9833        0.0492
          Context-enriched BM25 0.2096    0.2889       0.0578     0.3889        0.0389     0.4333        0.0217
Context-enriched Hybrid RRF 1:1 0.4585    0.7056       0.1411     0.9056        0.0906     0.9722        0.0486

Output directory:
/kaggle/working/BDLegalRAG/stage5_output

✓ All Stage 5 artifacts saved.
